In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


class TransformerEncoder(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):

        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError(
                "d_model must be divisible by num_heads"
            )

        # Multi-head self-attention
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout
        )

        # Feed-forward neural network
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model)
        ])

        # Normalization layers
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        # Dropout layers
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        # Query, key and value all come from x
        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            training=training
        )

        # Residual connection and normalization
        x = self.norm1(
            x + self.dropout1(
                attention_output,
                training=training
            )
        )

        # Feed-forward transformation
        ffn_output = self.ffn(x, training=training)

        # Second residual connection and normalization
        x = self.norm2(
            x + self.dropout2(
                ffn_output,
                training=training
            )
        )

        return x


# Example input:
# batch size = 2
# sequence length = 5
# embedding dimension = 32

sample = tf.random.normal((2, 5, 32))

encoder = TransformerEncoder(
    d_model=32,
    num_heads=4,
    ff_dim=64
)

output = encoder(sample)

print("Input shape:", sample.shape)
print("Output shape:", output.shape)

Input shape: (2, 5, 32)
Output shape: (2, 5, 32)
